# SAM2 Robot Arm Inference — Interactive Point Prompting

**Deployment notebook**: no GT masks needed — add point prompts on frame 0 and/or frame 45, and use all prompts together for propagation.

### Usage
1. Set paths in **Cell 1** and run Cells 1–3.
2. Run **Cell 4** — an interactive figure appears.
   - Choose **Prompt frame** (frame 0 / frame 45).
   - Toggle **ARM / GRIPPER** button to select the active object.
   - **Left-click** on the image to add a positive point (include in mask).
   - **Right-click** to add a negative point (exclude from mask).
   - Points are stored **per prompt frame** and preserved when switching frames.
3. When satisfied, run **Cell 5** to propagate through the entire video.
   - Prompts from **both frame 0 and frame 45** are jointly used for diffusion.
4. Run **Cell 6** to visualise key frames.
5. *(Optional)* Run **Cell 7** to save masks to disk.

> **Requires**: `pip install ipympl`  
> Restart the kernel after installing if it was just added.


In [ ]:
scid = 01
SCENE=f"scene_00{scid}"
SCENE_NAME = SCENE
OUT_DIR = f"/data/haoxiang/data/task0012_260321/task0012_toys_basket_converted/train/{SCENE}"   # change as needed
VIDEO_DIR = f"/data/haoxiang/data/task0012_260321/task0012_toys_basket_converted/train/{SCENE}/cam_104122060902/color"

In [ ]:
# Cell 1 — Imports & Paths
# ipympl enables interactive %matplotlib widget. Install once: pip install ipympl
%matplotlib widget
import os
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import torch
import ipywidgets as widgets
from IPython.display import display

# ── Adjust to your environment ─────────────────────────────────────────────
SAM2_ROOT = "/path/to/sam2_repo"   # e.g. "/home/haoxiang/sam2"
sys.path.insert(0, SAM2_ROOT)

CKPT_PATH = "/data/haoxiang/logs/sam2_finetune/airexo_task0013/checkpoints/checkpoint.pt"

# Directory containing per-frame jpg files named 00000.jpg, 00001.jpg, ...
# VIDEO_DIR = f"/data/haoxiang/data/airexo2/task_0012/sam2_symlinks/JPEGImages/{SCENE}"
# VIDEO_DIR = f"/data/haoxiang/data/task0012_260321/task0012_toys_basket_converted/train/{SCENE}/cam_104122060902"
# /data/haoxiang/data/task0012_260321/task0012_toys_basket_converted/train/{SCENE}/cam_104122060902
# /data/haoxiang/data/airexo2/task_0012/sam2_symlinks/JPEGImages/scene_0001
# VIDEO_DIR = "/data/haoxiang/data/airexo2_processed/sam2_finetune/JPEGImages/scene_0001"

# ──────────────────────────────────────────────────────────────────────────

print(f"Video dir : {VIDEO_DIR}")
print(f"Checkpoint: {CKPT_PATH}")
print(f"Checkpoint exists: {os.path.exists(CKPT_PATH)}")
frame_paths = sorted(glob.glob(os.path.join(VIDEO_DIR, "*.jpg")))
print(f"Frames found: {len(frame_paths)}")

# Cell 2 — Load fine-tuned model & initialise inference state
# NOTE: use the model-architecture config (sam2.1/), NOT the training config (sam2.1_training/)
from sam2.build_sam import build_sam2_video_predictor

predictor = build_sam2_video_predictor(
    config_file="configs/sam2.1/sam2.1_hiera_b+.yaml",
    ckpt_path=CKPT_PATH,
    device="cuda",
    mode="eval",
)

inference_state = predictor.init_state(video_path=VIDEO_DIR)
print("Model loaded, inference state initialised.")

In [ ]:
# Cell 4 — Interactive annotation on two prompt frames (frame 0 & frame 45)
#
# Controls:
#   [Prompt frame]          — choose frame 0 or frame 45 for annotation
#   [ARM / GRIPPER] toggle  — select which object to annotate
#   Left-click              — positive point (foreground)
#   Right-click             — negative point (background)
#   [Reset] button          — clear points on current prompt frame only
#
# Points are stored separately for each prompt frame and both can be used in Cell 5.

# Cell 3 — Helper functions

ARM_COLOR = (0.0, 0.6, 1.0)   # blue
GRP_COLOR = (1.0, 0.3, 0.0)   # orange
OBJ_COLORS = {1: ARM_COLOR, 2: GRP_COLOR}


def load_frame(video_dir, frame_idx):
    """Load RGB frame as np.ndarray (H, W, 3) uint8."""
    path = os.path.join(video_dir, f"{frame_idx:05d}.jpg")
    return np.array(Image.open(path).convert("RGB"))


def overlay_mask(image, mask, color, alpha=0.45):
    """Overlay a boolean mask on an RGB image with a given RGB color tuple (0-1 range)."""
    out = image.copy().astype(float)
    for c, v in enumerate(color[:3]):
        out[..., c] = np.where(mask, out[..., c] * (1 - alpha) + v * 255 * alpha, out[..., c])
    return out.clip(0, 255).astype(np.uint8)


def _new_obj_bucket():
    return {1: [], 2: []}


print("Helper functions defined.")

PROMPT_FRAME_CANDIDATES = [0, 20,45, 90]
AVAILABLE_PROMPT_FRAMES = [i for i in PROMPT_FRAME_CANDIDATES if i < len(frame_paths)]
if not AVAILABLE_PROMPT_FRAMES:
    raise RuntimeError("No valid prompt frame found (video has no frames).")

PROMPT_FRAME_IDX = AVAILABLE_PROMPT_FRAMES[0]
frame0 = load_frame(VIDEO_DIR, PROMPT_FRAME_IDX)
H, W = frame0.shape[:2]

# ── Shared state ──────────────────────────────────────────────────────────
click_state = {
    'points': {f: _new_obj_bucket() for f in AVAILABLE_PROMPT_FRAMES},
    'labels': {f: _new_obj_bucket() for f in AVAILABLE_PROMPT_FRAMES},
}
cached_masks = {
    f: {1: None, 2: None}
    for f in AVAILABLE_PROMPT_FRAMES
}

# ── Widgets ───────────────────────────────────────────────────────────────
frame_toggle = widgets.ToggleButtons(
    options=[(f"Frame {i}", i) for i in AVAILABLE_PROMPT_FRAMES],
    value=PROMPT_FRAME_IDX,
    description="Prompt frame:",
    style={"description_width": "initial"},
)
obj_toggle = widgets.ToggleButtons(
    options=[("ARM  (obj=1)", 1), ("GRIPPER  (obj=2)", 2)],
    value=1,
    description="Active object:",
    style={"button_width": "140px", "description_width": "initial"},
)
reset_btn = widgets.Button(
    description="Reset",
    button_style="danger",
    layout=widgets.Layout(width="100px"),
)
status_label = widgets.Label(value="Click on the image to add points.")

# ── Figure ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
plt.subplots_adjust(top=0.95)
ax.axis("off")


def _points(frame_idx, oid):
    return click_state['points'][frame_idx][oid]


def _labels(frame_idx, oid):
    return click_state['labels'][frame_idx][oid]


def _count_all_points(oid):
    return sum(len(_points(f, oid)) for f in AVAILABLE_PROMPT_FRAMES)


def _status_text():
    cur_arm = len(_points(PROMPT_FRAME_IDX, 1))
    cur_grp = len(_points(PROMPT_FRAME_IDX, 2))
    all_arm = _count_all_points(1)
    all_grp = _count_all_points(2)
    return (
        f"frame {PROMPT_FRAME_IDX}: arm={cur_arm}, gripper={cur_grp} | "
        f"all frames: arm={all_arm}, gripper={all_grp}"
    )


def _get_masks_for_current_frame():
    """Preview mask using prompts on current prompt frame only."""
    predictor.reset_state(inference_state)

    for o in [1, 2]:
        if _points(PROMPT_FRAME_IDX, o):
            pts = np.array(_points(PROMPT_FRAME_IDX, o), dtype=np.float32)
            lbs = np.array(_labels(PROMPT_FRAME_IDX, o), dtype=np.int32)
            with torch.inference_mode():
                predictor.add_new_points_or_box(
                    inference_state,
                    frame_idx=PROMPT_FRAME_IDX,
                    obj_id=o,
                    points=pts,
                    labels=lbs,
                )

    masks_out = {}
    for o in [1, 2]:
        if _points(PROMPT_FRAME_IDX, o):
            pts = np.array(_points(PROMPT_FRAME_IDX, o), dtype=np.float32)
            lbs = np.array(_labels(PROMPT_FRAME_IDX, o), dtype=np.int32)
            with torch.inference_mode():
                _, obj_ids, mask_logits = predictor.add_new_points_or_box(
                    inference_state,
                    frame_idx=PROMPT_FRAME_IDX,
                    obj_id=o,
                    points=pts,
                    labels=lbs,
                )
            if o in obj_ids:
                idx = list(obj_ids).index(o)
                masks_out[o] = mask_logits[idx].squeeze().cpu().numpy() > 0.0
    return masks_out


def refresh_display():
    """Redraw the figure with current masks and point markers."""
    ax.clear()
    ax.axis("off")

    composite = frame0.copy()
    masks = _get_masks_for_current_frame()
    for oid, color in OBJ_COLORS.items():
        if oid in masks:
            cached_masks[PROMPT_FRAME_IDX][oid] = masks[oid]
        if cached_masks[PROMPT_FRAME_IDX][oid] is not None:
            composite = overlay_mask(composite, cached_masks[PROMPT_FRAME_IDX][oid], color)

    ax.imshow(composite)

    for oid, color in OBJ_COLORS.items():
        for (x, y), lbl in zip(_points(PROMPT_FRAME_IDX, oid), _labels(PROMPT_FRAME_IDX, oid)):
            marker = "*" if lbl == 1 else "x"
            msize  = 14  if lbl == 1 else 12
            mew    = 1.5 if lbl == 1 else 2.5
            ax.plot(
                x, y, marker,
                color=color,
                markersize=msize,
                markeredgewidth=mew,
                markeredgecolor="white",
            )

    legend_handles = [
        mpatches.Patch(color=ARM_COLOR, label="arm (obj=1)"),
        mpatches.Patch(color=GRP_COLOR, label="gripper (obj=2)"),
    ]
    ax.legend(handles=legend_handles, loc="upper right", fontsize=10)
    ax.set_title(f"Prompt Frame {PROMPT_FRAME_IDX}  |  ★ = positive   ✕ = negative", fontsize=11)
    fig.canvas.draw_idle()


def on_click(event):
    if event.inaxes != ax or event.xdata is None:
        return
    if event.button not in (1, 3):
        return

    oid = obj_toggle.value
    lbl = 1 if event.button == 1 else 0   # left = pos, right = neg
    _points(PROMPT_FRAME_IDX, oid).append([event.xdata, event.ydata])
    _labels(PROMPT_FRAME_IDX, oid).append(lbl)

    status_label.value = _status_text()
    refresh_display()


def on_reset(_):
    click_state['points'][PROMPT_FRAME_IDX] = _new_obj_bucket()
    click_state['labels'][PROMPT_FRAME_IDX] = _new_obj_bucket()
    cached_masks[PROMPT_FRAME_IDX] = {1: None, 2: None}
    predictor.reset_state(inference_state)

    ax.clear()
    ax.axis("off")
    ax.imshow(frame0)
    ax.set_title(f"Prompt Frame {PROMPT_FRAME_IDX}  |  ★ = positive   ✕ = negative", fontsize=11)
    status_label.value = f"Cleared points on frame {PROMPT_FRAME_IDX}. " + _status_text()
    fig.canvas.draw_idle()


def on_frame_change(change):
    if change["name"] != "value":
        return
    if change["new"] == change.get("old", None):
        return

    global PROMPT_FRAME_IDX, frame0, H, W
    PROMPT_FRAME_IDX = int(change["new"])
    frame0 = load_frame(VIDEO_DIR, PROMPT_FRAME_IDX)
    H, W = frame0.shape[:2]

    predictor.reset_state(inference_state)
    refresh_display()
    status_label.value = f"Switched to frame {PROMPT_FRAME_IDX}. " + _status_text()


fig.canvas.mpl_connect("button_press_event", on_click)
reset_btn.on_click(on_reset)
frame_toggle.observe(on_frame_change, names="value")

# Initial render
ax.imshow(frame0)
ax.set_title(f"Prompt Frame {PROMPT_FRAME_IDX}  |  ★ = positive   ✕ = negative", fontsize=11)
status_label.value = _status_text()

display(
    widgets.VBox([
        widgets.HBox([frame_toggle, obj_toggle]),
        widgets.HBox([reset_btn, status_label]),
        fig.canvas,
    ])
)


In [ ]:
# Cell 5 — Propagate through the entire video
# Run this AFTER you are happy with the points in Cell 4.


def _annotated_prompt_frames():
    return [
        f for f in AVAILABLE_PROMPT_FRAMES
        if any(len(click_state['points'][f][oid]) > 0 for oid in [1, 2])
    ]


annotated_frames = sorted(_annotated_prompt_frames())
assert annotated_frames, "No points found. Please click on the image in Cell 4 first."
print(f"Annotated prompt frames: {annotated_frames}")


def _seed_all_prompts():
    """Reset inference state and add all prompts from all annotated prompt frames."""
    predictor.reset_state(inference_state)
    for fidx in annotated_frames:
        for oid in [1, 2]:
            pts = click_state['points'][fidx][oid]
            if not pts:
                continue
            with torch.inference_mode():
                predictor.add_new_points_or_box(
                    inference_state,
                    frame_idx=fidx,
                    obj_id=oid,
                    points=np.array(pts, dtype=np.float32),
                    labels=np.array(click_state['labels'][fidx][oid], dtype=np.int32),
                )


def _run_propagation(start_frame_idx, reverse=False, max_frame_num_to_track=None):
    out = {}
    with torch.inference_mode():
        for f_idx, obj_ids, mask_logits in predictor.propagate_in_video(
            inference_state,
            start_frame_idx=start_frame_idx,
            max_frame_num_to_track=max_frame_num_to_track,
            reverse=reverse,
        ):
            out[f_idx] = {
                oid: (mask_logits[i].squeeze().cpu().numpy() > 0.0)
                for i, oid in enumerate(obj_ids)
            }
    return out


# Use all prompts together; propagate from earliest annotated frame.
start_idx = min(annotated_frames)
print(f"Propagation start frame: {start_idx}")

_seed_all_prompts()
all_masks = _run_propagation(start_frame_idx=start_idx, reverse=False, max_frame_num_to_track=None)

# If earliest prompt frame > 0, add a reverse pass to cover [0, start_idx].
if start_idx > 0:
    _seed_all_prompts()
    backward_masks = _run_propagation(
        start_frame_idx=start_idx,
        reverse=True,
        max_frame_num_to_track=start_idx + 1,
    )
    all_masks.update(backward_masks)
    print(f"Bidirectional propagation enabled (forward + backward from frame {start_idx}).")
else:
    print("Forward-only propagation (earliest prompt frame is 0).")

print(f"Propagated {len(all_masks)} frames.")
print(f"Objects tracked: {sorted({oid for m in all_masks.values() for oid in m})}")

# Cell 6 — Visualise key frames
%matplotlib inline
import matplotlib
matplotlib.rcParams["figure.dpi"] = 100

n_frames  = len(all_masks)
vis_idxs  = sorted(all_masks.keys())
step      = max(1, n_frames // 4)
VIS_FRAMES = [vis_idxs[0], vis_idxs[step], vis_idxs[2 * step], vis_idxs[-1]]

n_rows = len(VIS_FRAMES)
fig, axes = plt.subplots(n_rows, 2, figsize=(10, 3.5 * n_rows))
if n_rows == 1:
    axes = axes[np.newaxis, :]

for row, f_idx in enumerate(VIS_FRAMES):
    img      = load_frame(VIDEO_DIR, f_idx)
    arm_pred = all_masks.get(f_idx, {}).get(1, np.zeros(img.shape[:2], bool))
    grp_pred = all_masks.get(f_idx, {}).get(2, np.zeros(img.shape[:2], bool))

    axes[row, 0].imshow(overlay_mask(img, arm_pred, ARM_COLOR))
    axes[row, 0].set_title(f"Frame {f_idx} — arm (pred)")
    axes[row, 1].imshow(overlay_mask(img, grp_pred, GRP_COLOR))
    axes[row, 1].set_title(f"Frame {f_idx} — gripper (pred)")

for ax_ in axes.flatten():
    ax_.axis("off")

legend = [
    mpatches.Patch(color=ARM_COLOR, label="arm"),
    mpatches.Patch(color=GRP_COLOR, label="gripper"),
]
fig.legend(handles=legend, loc="lower center", ncol=2, fontsize=12)
plt.suptitle(f"{os.path.basename(VIDEO_DIR)} — SAM2 point-prompted", fontsize=14)
plt.tight_layout(rect=[0, 0.04, 1, 0.98])
plt.show()

# Cell 7 (optional) — Save masks to disk

# OUT_DIR = f"/data/haoxiang/data/airexo2/task_0012/masks/{SCENE}"   # change as needed
os.makedirs(os.path.join(OUT_DIR, "arm"),     exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "gripper"), exist_ok=True)

for f_idx, masks in sorted(all_masks.items()):
    h_ref, w_ref = frame0.shape[:2]
    arm_mask = (masks.get(1, np.zeros((h_ref, w_ref), bool)) * 255).astype(np.uint8)
    grp_mask = (masks.get(2, np.zeros((h_ref, w_ref), bool)) * 255).astype(np.uint8)
    Image.fromarray(arm_mask).save(os.path.join(OUT_DIR, "arm",     f"{f_idx:05d}.png"))
    Image.fromarray(grp_mask).save(os.path.join(OUT_DIR, "gripper", f"{f_idx:05d}.png"))

print(f"Saved {len(all_masks)} frames to {OUT_DIR}/arm/ and {OUT_DIR}/gripper/")

# ---

# imports

# ============================================================
# Imports
# ============================================================
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import yaml
from PIL import Image

from mask_pipeline_tools import (
    get_frame_size,
    load_video_frames_for_visualization,
)

plt.rcParams["axes.titlesize"] = 12
plt.rcParams["figure.titlesize"] = 12

# 配置加载

# ============================================================
# 配置加载 — 修改 mask_config.yaml 来调整参数
# ============================================================
import yaml

_CONFIG_PATH = Path("mask_config.yaml")
with open(_CONFIG_PATH, "r") as _f:
    _cfg = yaml.safe_load(_f)

TASK_NAME = _cfg['task_name']
# SCENE_NAME = _cfg['scene_name']
CHECKPOINT_PATH = _cfg.get('sam2_checkpoint', _cfg.get('sam3_checkpoint'))
CUDA_VISIBLE_DEVICES = _cfg['cuda_visible_devices']
APPLY_TEMPORAL_DISAMBIGUATION = bool(_cfg['apply_temporal_disambiguation'])
ARM_OBJ_ID = int(_cfg['arm_obj_id'])
ARM_OBJ_ID_2 = int(_cfg['arm_obj_id_2']) if _cfg.get('arm_obj_id_2') is not None else None
GRIPPER_LEFT_OBJ_ID = int(_cfg['gripper_left_obj_id'])
GRIPPER_RIGHT_OBJ_ID = int(_cfg['gripper_right_obj_id'])
ARM_TEXT_PROMPT = _cfg['arm_text_prompt']
ARM_TEXT_BOOTSTRAP_FRAME_INDEX = int(_cfg['arm_text_bootstrap_frame_index'])
VIS_FRAME_STRIDE = int(_cfg['vis_frame_stride'])
VIS_MAX_PLOTS = int(_cfg['vis_max_plots'])
EXPORT_ARM_DILATE_RADIUS = int(_cfg['export_arm_dilate_radius'])
EXPORT_GRIPPER_DILATE_RADIUS = int(_cfg['export_gripper_dilate_radius'])
EXPORT_LOG_EVERY = int(_cfg['export_log_every'])

# Derived paths
_data_base = _cfg['data_base_dir']
_export_base_str = _cfg['export_base_dir']
VIDEO_PATH = f"{_data_base}/{TASK_NAME}/train/{SCENE_NAME}/cam_105422061350/color"
ANNOTATION_JSON_PATH = str(Path(VIDEO_PATH).parent / "annotation_prompts_gripper_points.json")
ARM_ANNOTATION_JSON_PATH = str(Path(VIDEO_PATH).parent / "annotation_prompts_arm_points.json")
_export_base = Path(_export_base_str)

def _has_arm_gripper_dir(_p: Path) -> bool:
    return (_p / "arm").is_dir() and (_p / "gripper").is_dir()

# 1) 解析 checkpoint 输入目录（来自生成代码 OUT_DIR/{arm,gripper}）
_ckpt_root_candidates = [
    Path(_data_base) / TASK_NAME / "masks",
    _export_base.parent,
    _export_base,
    Path(_data_base) / "masks",
]
_ckpt_roots = []
for _r in _ckpt_root_candidates:
    if _r not in _ckpt_roots:
        _ckpt_roots.append(_r)

_resolved_ckpt_scene_dir = None
for _r in _ckpt_roots:
    _cand = _r / SCENE_NAME
    if _has_arm_gripper_dir(_cand):
        _resolved_ckpt_scene_dir = _cand
        break

if _resolved_ckpt_scene_dir is None:
    _hits = []
    for _r in _ckpt_roots:
        if not _r.exists():
            continue
        for _d in sorted(_r.glob("scene_*")):
            if _has_arm_gripper_dir(_d) and _d not in _hits:
                _hits.append(_d)
    if len(_hits) >= 1:
        _hits = sorted(_hits, key=lambda p: p.stat().st_mtime, reverse=True)
        _resolved_ckpt_scene_dir = _hits[0]
        if SCENE_NAME != _resolved_ckpt_scene_dir.name:
            print(f"[config][warn] SCENE_NAME={SCENE_NAME} 未命中，自动回退到最近更新场景: {_resolved_ckpt_scene_dir.name}")
            print(f"[config][warn] 可用场景: {[p.name for p in _hits]}")
            SCENE_NAME = _resolved_ckpt_scene_dir.name
    else:
        _resolved_ckpt_scene_dir = (_ckpt_roots[0] / SCENE_NAME)

CHECKPOINT_ARM_DIR     = str(_resolved_ckpt_scene_dir / "arm")
CHECKPOINT_GRIPPER_DIR = str(_resolved_ckpt_scene_dir / "gripper")

# 2) 解析 merge 输出目录（与 checkpoint 输入目录解耦）
_out_base = _export_base
if _out_base.name.startswith("scene_"):
    EXPORT_OUTPUT_DIR = str(_out_base)
else:
    EXPORT_OUTPUT_DIR = str(_out_base / SCENE_NAME)

# 若上面自动纠正了 SCENE_NAME，这里同步更新依赖路径
VIDEO_PATH = f"{_data_base}/{TASK_NAME}/train/{SCENE_NAME}/cam_105422061350/color"
ANNOTATION_JSON_PATH = str(Path(VIDEO_PATH).parent / "annotation_prompts_gripper_points.json")
ARM_ANNOTATION_JSON_PATH = str(Path(VIDEO_PATH).parent / "annotation_prompts_arm_points.json")

print(f"[config] task={TASK_NAME}  scene={SCENE_NAME}")
print(f"[config] VIDEO_PATH={VIDEO_PATH}")
print(f"[config] ANNOTATION_JSON_PATH={ANNOTATION_JSON_PATH}")
print(f"[config] ARM_ANNOTATION_JSON_PATH={ARM_ANNOTATION_JSON_PATH}")
print(f"[config] CHECKPOINT_ARM_DIR={CHECKPOINT_ARM_DIR}")
print(f"[config] CHECKPOINT_GRIPPER_DIR={CHECKPOINT_GRIPPER_DIR}")
print(f"[config] EXPORT_OUTPUT_DIR={EXPORT_OUTPUT_DIR}")

# 初始化

# ============================================================
# 加载视频帧（仅用于可视化及获取帧尺寸）
# ============================================================
video_frames_for_vis = load_video_frames_for_visualization(VIDEO_PATH)
TOTAL_FRAMES = len(video_frames_for_vis)
IMG_WIDTH, IMG_HEIGHT = get_frame_size(video_frames_for_vis)

print(f'[init] frames={TOTAL_FRAMES}  size={IMG_WIDTH}x{IMG_HEIGHT}')
print(f'[init] CHECKPOINT_ARM_DIR={CHECKPOINT_ARM_DIR}')
print(f'[init] CHECKPOINT_GRIPPER_DIR={CHECKPOINT_GRIPPER_DIR}')
print(f'[init] EXPORT_OUTPUT_DIR={EXPORT_OUTPUT_DIR}')


# Merge + 导出 arm_only 掩膜

# ============================================================
# Merge + 导出 arm_only 掩膜
# 逻辑：先分别膨胀 arm / gripper，再 boolean 相减
#
# 数据来源：仅从 checkpoint PNG 读取（不依赖 Session 内存）
#   arm:     CHECKPOINT_ARM_DIR（由 mask_1_arm.ipynb 生成）
#   gripper: CHECKPOINT_GRIPPER_DIR（由 mask_2_gripper.ipynb 生成）
# ============================================================

# --- 校验 checkpoint 目录 ---
_arm_ckpt_files = sorted(os.listdir(CHECKPOINT_ARM_DIR)) if os.path.isdir(CHECKPOINT_ARM_DIR) else []
_gri_ckpt_files = sorted(os.listdir(CHECKPOINT_GRIPPER_DIR)) if os.path.isdir(CHECKPOINT_GRIPPER_DIR) else []

if not _arm_ckpt_files:
    raise RuntimeError(
        f"[merge] arm checkpoint 不存在或为空: {CHECKPOINT_ARM_DIR}\n"
        "请先运行 mask_1_arm.ipynb。"
    )
if not _gri_ckpt_files:
    raise RuntimeError(
        f"[merge] gripper checkpoint 不存在或为空: {CHECKPOINT_GRIPPER_DIR}\n"
        "请先运行 mask_2_gripper.ipynb。"
    )

print(f"[merge] arm checkpoint:     {len(_arm_ckpt_files)} frames ← {CHECKPOINT_ARM_DIR}")
print(f"[merge] gripper checkpoint: {len(_gri_ckpt_files)} frames ← {CHECKPOINT_GRIPPER_DIR}")
print(f"[merge] arm_dilate_radius={EXPORT_ARM_DILATE_RADIUS}  gripper_dilate_radius={EXPORT_GRIPPER_DILATE_RADIUS}")
print(f"[merge] output_dir={EXPORT_OUTPUT_DIR}")

os.makedirs(EXPORT_OUTPUT_DIR, exist_ok=True)

def _make_kernel(radius):
    if radius <= 0:
        return None
    ks = 2 * int(radius) + 1
    return np.ones((ks, ks), np.uint8)

_arm_kernel     = _make_kernel(EXPORT_ARM_DILATE_RADIUS)
_gripper_kernel = _make_kernel(EXPORT_GRIPPER_DILATE_RADIUS)
print(f"[merge] arm kernel: {2*EXPORT_ARM_DILATE_RADIUS+1}x{2*EXPORT_ARM_DILATE_RADIUS+1}  "
      f"gripper kernel: {2*EXPORT_GRIPPER_DILATE_RADIUS+1}x{2*EXPORT_GRIPPER_DILATE_RADIUS+1}")

_img_w, _img_h = get_frame_size(video_frames_for_vis)

# --- 主循环 ---
_saved_paths = []
_total_gripper_px = 0

for _frame_idx in range(TOTAL_FRAMES):
    _arm_union = np.array(
        Image.open(os.path.join(CHECKPOINT_ARM_DIR, f"{_frame_idx:05d}.png")).convert("L")
    )
    _gripper_union = np.array(
        Image.open(os.path.join(CHECKPOINT_GRIPPER_DIR, f"{_frame_idx:05d}.png")).convert("L")
    )

    # Step 1: 分别膨胀
    if _arm_kernel is not None and np.any(_arm_union):
        _arm_union = cv2.dilate(_arm_union, _arm_kernel, iterations=1)
    if _gripper_kernel is not None and np.any(_gripper_union):
        _gripper_union = cv2.dilate(_gripper_union, _gripper_kernel, iterations=1)

    # Step 2: boolean 相减（dilated_arm AND NOT dilated_gripper）
    _arm_only = np.where((_arm_union > 0) & (_gripper_union == 0), 255, 0).astype(np.uint8)

    _total_gripper_px += int(np.count_nonzero(_gripper_union))

    if _frame_idx % max(int(EXPORT_LOG_EVERY), 1) == 0 or _frame_idx == TOTAL_FRAMES - 1:
        print(
            f"[merge][frame {_frame_idx:05d}] "
            f"arm_px={int(np.count_nonzero(_arm_union))} "
            f"gripper_px={int(np.count_nonzero(_gripper_union))} "
            f"arm_only_px={int(np.count_nonzero(_arm_only))}"
        )

    _out_fp = os.path.join(EXPORT_OUTPUT_DIR, f"{_frame_idx:05d}.png")
    Image.fromarray(_arm_only, mode="L").save(_out_fp)
    _saved_paths.append(_out_fp)

if _total_gripper_px == 0:
    raise ValueError(
        "[merge][FATAL] 所有帧的 gripper 像素总和为 0。\n"
        "checkpoint 内容可能无效。请检查 mask_2_gripper.ipynb 的分割结果。"
    )

print(f"[merge] export complete: {len(_saved_paths)} masks → {EXPORT_OUTPUT_DIR}")

# ============================================================
# 可视化 Merge 结果（抽帧叠加检查）
# 左：原图   右：arm_only mask 叠加（绿色 = mask 区域）
# ============================================================

_vis_frames = list(range(0, TOTAL_FRAMES, VIS_FRAME_STRIDE))[:VIS_MAX_PLOTS]
plt.close("all")

for _fidx in _vis_frames:
    _mask_fp = os.path.join(EXPORT_OUTPUT_DIR, f"{_fidx:05d}.png")
    if not os.path.exists(_mask_fp):
        print(f"[vis/merge] mask not found: {_mask_fp}")
        continue

    _mask_img = np.array(Image.open(_mask_fp).convert("L"))
    _frame_img = video_frames_for_vis[_fidx].copy()

    _overlay = _frame_img.copy()
    _mb = _mask_img > 0
    _overlay[_mb] = (
        _frame_img[_mb] * 0.4 + np.array([0, 220, 0], dtype=np.float32) * 0.6
    ).astype(np.uint8)

    _fig, _axes = plt.subplots(1, 2, figsize=(13, 4))
    _axes[0].imshow(_frame_img)
    _axes[0].set_title(f"Frame {_fidx}: original")
    _axes[0].axis("off")
    _axes[1].imshow(_overlay)
    _axes[1].set_title(f"Frame {_fidx}: arm_only overlay")
    _axes[1].axis("off")
    plt.tight_layout()
    plt.show()

